In [1]:
import numpy as np

# ---------- 1. Define the GridWorld ----------
GRID_SIZE = 5
GAMMA = 0.9          # discount factor
THETA = 1e-4         # convergence threshold

In [2]:
grid_layout = [
    ['S', '.', '.', '.', '.'],
    ['.', 'H', '.', 'H', '.'],
    ['.', '.', '.', '.', '.'],
    ['.', 'H', '.', '.', '.'],
    ['.', '.', '.', 'H', 'G'],
]

ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']
ACTION_DELTA = {
    'UP': (-1, 0),
    'DOWN': (1, 0),
    'LEFT': (0, -1),
    'RIGHT': (0, 1),
}

In [4]:
def is_terminal(r, c):
    return grid_layout[r][c] in ('H', 'G')


In [5]:
def reward(r, c):
    cell = grid_layout[r][c]
    if cell == 'G':
        return 1.0
    if cell == 'H':
        return -1.0
    return -0.04  # small step cost so the agent prefers shorter paths

In [6]:
def next_state(r, c, action):
    """Deterministic transition. Bumping into a wall keeps the agent in place."""
    if is_terminal(r, c):
        return r, c
    dr, dc = ACTION_DELTA[action]
    nr, nc = r + dr, c + dc
    if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
        return nr, nc
    return r, c  # wall bump

In [7]:
# ---------- 2. Value Iteration ----------
def value_iteration():
    V = np.zeros((GRID_SIZE, GRID_SIZE))
    iteration = 0
    while True:
        delta = 0.0
        new_V = V.copy()
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                if is_terminal(r, c):
                    new_V[r, c] = reward(r, c)
                    continue
                action_values = []
                for a in ACTIONS:
                    nr, nc = next_state(r, c, a)
                    q_sa = reward(r, c) + GAMMA * V[nr, nc]
                    action_values.append(q_sa)
                new_V[r, c] = max(action_values)
                delta = max(delta, abs(new_V[r, c] - V[r, c]))
        V = new_V
        iteration += 1
        if delta < THETA:
            break
    return V, iteration

In [8]:
def extract_policy(V):
    policy = np.full((GRID_SIZE, GRID_SIZE), ' ', dtype='<U5')
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if is_terminal(r, c):
                policy[r, c] = grid_layout[r][c]
                continue
            best_a, best_val = None, -np.inf
            for a in ACTIONS:
                nr, nc = next_state(r, c, a)
                q_sa = reward(r, c) + GAMMA * V[nr, nc]
                if q_sa > best_val:
                    best_val = q_sa
                    best_a = a
            policy[r, c] = best_a
    return policy

In [9]:
if __name__ == "__main__":
    V, num_iterations = value_iteration()
    policy = extract_policy(V)

    print(f"Converged in {num_iterations} iterations\n")
    print("Final Value Table (rounded to 2 decimals):")
    print(np.round(V, 2))
    print("\nExtracted Optimal Policy:")
    for row in policy:
        print(' '.join(f"{cell:>5}" for cell in row))

Converged in 10 iterations

Final Value Table (rounded to 2 decimals):
[[ 0.2   0.27  0.34  0.43  0.52]
 [ 0.27 -1.    0.43 -1.    0.62]
 [ 0.34  0.43  0.52  0.62  0.73]
 [ 0.27 -1.    0.62  0.73  0.86]
 [ 0.34  0.43  0.52 -1.    1.  ]]

Extracted Optimal Policy:
 DOWN RIGHT  DOWN RIGHT  DOWN
 DOWN     H  DOWN     H  DOWN
RIGHT RIGHT  DOWN  DOWN  DOWN
   UP     H RIGHT RIGHT  DOWN
RIGHT RIGHT    UP     H     G
